In [1]:
import os
os.chdir("C:/journal_project")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, f1_score

In [2]:
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=["hr", "hrv", "co2_noisy", "posture_cm"])
y = (df["ground_truth"] == "STRESSED").astype(int).values
groups = df["subject"].values
hr = df["hr"].values.reshape(-1, 1)
X_full = df[["hr", "hrv", "co2_noisy", "posture_cm"]].values

In [3]:
N_REPEATS = 10
youden_f1, tree_f1, fixed_f1 = [], [], []

for repeat in range(N_REPEATS):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for train_idx, test_idx in sgkf.split(hr, y, groups=groups):
        # Youden's J baseline
        fpr, tpr, thresholds = roc_curve(y[train_idx], hr[train_idx].ravel())
        best_thresh = thresholds[np.argmax(tpr - fpr)]
        preds_j = (hr[test_idx].ravel() > best_thresh).astype(int)
        youden_f1.append(f1_score(y[test_idx], preds_j, average="macro", zero_division=0))

        # Depth-2 tree baseline
        tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
        tree.fit(X_full[train_idx], y[train_idx])
        preds_tree = tree.predict(X_full[test_idx])
        tree_f1.append(f1_score(y[test_idx], preds_tree, average="macro", zero_division=0))

        # Original fixed-threshold rule engine, for direct comparison
        preds_fixed = ((df["hrv"].values[test_idx] < 25) & (df["hr"].values[test_idx] > 90)).astype(int)
        fixed_f1.append(f1_score(y[test_idx], preds_fixed, average="macro", zero_division=0))

print(f"Fixed threshold:  F1 = {np.mean(fixed_f1):.4f}")
print(f"Youden's J:       F1 = {np.mean(youden_f1):.4f}")
print(f"Depth-2 tree:     F1 = {np.mean(tree_f1):.4f}")

Fixed threshold:  F1 = 0.6676
Youden's J:       F1 = 0.7306
Depth-2 tree:     F1 = 0.6812


In [4]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
# youden_f1 is already in memory from Cell 3 above

def corrected_resampled_ttest(scores_a, scores_b, n_train, n_test):
    diffs = np.array(scores_a) - np.array(scores_b)
    n = len(diffs)
    mean_diff = np.mean(diffs)
    var_diff = np.var(diffs, ddof=1)
    correction = (1 / n) + (n_test / n_train)
    corrected_var = var_diff * correction
    t_stat = mean_diff / np.sqrt(corrected_var)
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 1))
    return t_stat, p_value

from scipy import stats
n_test = len(df) // 5
n_train = len(df) - n_test
t_stat, p_val = corrected_resampled_ttest(scores_df["Proposed_Stacking_Ensemble"], youden_f1, n_train, n_test)
print(f"Stacking Ensemble vs Youden's J baseline: corrected t={t_stat:.3f}, p={p_val:.4f}")

Stacking Ensemble vs Youden's J baseline: corrected t=0.685, p=0.4965


In [5]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
t_stat, p_val = corrected_resampled_ttest(scores_df["XGBoost"], youden_f1, n_train, n_test)
print(f"XGBoost vs Youden's J baseline: corrected t={t_stat:.3f}, p={p_val:.4f}")

XGBoost vs Youden's J baseline: corrected t=0.238, p=0.8132


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"]
df_cascade = df.dropna(subset=FEATURE_COLS)
X_cascade = df_cascade[FEATURE_COLS].values
y_cascade = (df_cascade["ground_truth"] == "STRESSED").astype(int).values
groups_cascade = df_cascade["subject"].values

def run_cascade_at_margin(margin):
    fold_f1, ml_pct = [], []
    for repeat in range(10):
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for train_idx, test_idx in sgkf.split(X_cascade, y_cascade, groups=groups_cascade):
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(X_cascade[train_idx])
            Xte = scaler.transform(X_cascade[test_idx])
            clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
            clf.fit(Xtr, y_cascade[train_idx])
            ml_preds = clf.predict(Xte)

            hr_test = df_cascade["hr"].values[test_idx]
            hrv_test = df_cascade["hrv"].values[test_idx]
            rule_preds = ((hrv_test < 25) & (hr_test > 90)).astype(int)
            is_ambiguous = (np.abs(hr_test - 90) <= margin) | (np.abs(hrv_test - 25) <= margin)

            cascade_preds = np.where(is_ambiguous, ml_preds, rule_preds)
            fold_f1.append(f1_score(y_cascade[test_idx], cascade_preds, average="macro", zero_division=0))
            ml_pct.append(is_ambiguous.mean() * 100)
    return np.mean(fold_f1), np.mean(ml_pct)

for margin in [2, 5, 10]:
    f1, pct = run_cascade_at_margin(margin)
    print(f"Margin={margin}: Cascade F1={f1:.4f}, ML invoked on {pct:.1f}% of samples")

Margin=2: Cascade F1=0.6979, ML invoked on 12.7% of samples
Margin=5: Cascade F1=0.7207, ML invoked on 23.9% of samples
Margin=10: Cascade F1=0.7525, ML invoked on 43.6% of samples
